# EVASPA zones

### Imports

In [ ]:
from __future__ import annotations

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

import contextily as cx
import geopandas as gpd
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import xarray as xr
from matplotlib.colors import ListedColormap
from rasterio.features import geometry_mask

In [ ]:
if os.environ["EVASPA_TEST_DATA_PATH"] is None:
    msg = "Variable EVASPA_TEST_DATA_PATH must be set"
    raise ValueError(msg)
if os.environ["MNT_PATH"] is None:
    os.environ["MNT_PATH"] = os.environ["EVASPA_TEST_DATA_PATH"]

In [ ]:
from evaspa import trishna, zones
from evaspa.dem import get_dem_from_tiles

In [ ]:
outdir = "../report"
os.makedirs(outdir, exist_ok=True)

### Data

In [ ]:
land = trishna.get_land_mask()
tiles = gpd.read_file("roi_italy.gpkg")

In [ ]:
tiles.crs

### Define valid pixels in a tile

In [ ]:
def plot_valid_pixels(
    tile_ids: list[str],
    land: gpd.GeoDataFrame,
    slope_threshold: float = 30,
    range_threshold: int = 300,
    simplify: bool = True,
) -> None:
    """
    Plot valid pixels
    """
    # Read DEM
    dem = get_dem_from_tiles(tile_ids)
    # Compute valid pixels
    pvalid, dem = zones.define_valid_pixels(
        dem,
        land,
        slope_threshold=slope_threshold,
        range_threshold=range_threshold,
        simplify=simplify,
    )
    # Plot
    valid = dem["valid"]
    ax = plt.gca()
    valid_cmap = ListedColormap([(1.0, 0.0, 0.0, 1.0), (0.0, 0.0, 0.0, 0.0)])
    red_patch = mpatches.Patch(color="red", label="Invalid pixels")
    fg = dem["height"].plot(
        ax=ax,
        vmin=dem["height"].min(),
        vmax=dem["height"].max(),
        add_colorbar=False,
    )
    valid.plot(ax=ax, cmap=valid_cmap, add_colorbar=False)
    plt.title(
        f"{'/'.join(tile_ids)} - Slope < {slope_threshold} - Range = {range_threshold} {' - cleaning ' if simplify else ''}\n Percentage of valid pixels = {pvalid * 100:.2f}",
        fontsize=18,
    )
    ax.tick_params(axis="both", which="major", labelsize=12)
    ax.set_xlabel("x", fontsize=15)
    ax.set_ylabel("y", fontsize=15)
    cb = plt.colorbar(fg, orientation="vertical")
    cb.set_label(label="Height (m)", size=17)
    cb.ax.tick_params(labelsize=12)
    plt.legend(handles=[red_patch], loc="lower right", fontsize=16)
    plt.show()

In [ ]:
plot_valid_pixels(["31TCJ"], land, range_threshold=300)

In [ ]:
def plot_tile_figs(
    tile_ids: list[str],
    land: gpd.GeoDataFrame,
) -> None:
    """
    Plot valid pixels
    """
    simplify = True
    # Read DEM
    dem = get_dem_from_tiles(tile_ids)
    # Compute various case
    results = []
    for slope_threshold in [15, 30]:
        for range_threshold in [300, 600]:
            # Compute valid pixels
            pvalid, res = zones.define_valid_pixels(
                dem,
                land,
                slope_threshold=slope_threshold,
                range_threshold=range_threshold,
                simplify=simplify,
            )
            # Save result: slope, range, pvalid and valid
            results.append(
                [slope_threshold, range_threshold, pvalid, res["valid"].copy()]
            )

    # Plot
    fig, axs = plt.subplots(
        ncols=5, nrows=1, figsize=(30, 5), layout="constrained"
    )
    valid_cmap = ListedColormap([(1.0, 0.0, 0.0, 1.0), (0.0, 0.0, 0.0, 0.0)])
    red_patch = mpatches.Patch(color="red", label="Invalid pixels")
    plot_bar = False
    # DEM
    ax = axs[0]
    dem["height"].plot(
        ax=ax,
        vmin=dem["height"].min(),
        vmax=dem["height"].max(),
        add_colorbar=plot_bar,
    )
    ax.set_title("DEM", fontsize=20)
    ax.tick_params(axis="both", which="major", labelsize=12)
    ax.set_xlabel("x", fontsize=15)
    ax.set_ylabel("y", fontsize=15)
    # Result
    for i, (ax, res) in enumerate(zip(axs[1:], results, strict=False)):
        fg = dem["height"].plot(
            ax=ax,
            vmin=dem["height"].min(),
            vmax=dem["height"].max(),
            add_colorbar=plot_bar,
        )
        res[3].plot(ax=ax, cmap=valid_cmap, add_colorbar=False)
        ax.set_title(
            f"Slope < {res[0]} - Range = {res[1]}m\n Percentage of valid pixels = {res[2] * 100:.2f}",
            fontsize=20,
        )
        ax.legend(handles=[red_patch], loc="lower right", fontsize=16)
        ax.tick_params(axis="both", which="major", labelsize=12)
        ax.set_xlabel("x", fontsize=15)
        ax.set_ylabel("y", fontsize=15)
        if i == 3:
            cb = plt.colorbar(fg, orientation="vertical")
            cb.set_label(label="Height (m)", size=17)
            cb.ax.tick_params(labelsize=12)

    fig.suptitle(f"{'/'.join(tile_ids)}", fontsize=32)
    fig.savefig(os.path.join(outdir, f"plot_{'-'.join(tile_ids)}.png"), dpi=600)
    plt.show()

In [ ]:
plot_tile_figs(["32TLP"], land)

In [ ]:
def plot_valid_zones(
    tile_ids: list[str],
    land: gpd.GeoDataFrame,
    slope_threshold: float = 30,
    range_threshold: int = 300,
    simplify: bool = True,
) -> None:
    # Read DEM
    dem = get_dem_from_tiles(tile_ids)
    # Compute zones
    pvalid, nbvalid, polys = zones.define_valid_zones(
        tile_ids,
        land,
        slope_threshold=slope_threshold,
        range_threshold=range_threshold,
        simplify=simplify,
    )
    # Transform polygons to mask
    valid = xr.DataArray(
        geometry_mask(
            polys.geoms,
            out_shape=(dem.sizes["y"], dem.sizes["x"]),
            transform=dem.transform,
        ),
        coords=dem.coords,
        dims=("y", "x"),
    )
    valid.data = ~valid.data
    # Plot
    ax = plt.gca()
    valid_cmap = ListedColormap([(1.0, 0.0, 0.0, 1.0), (0.0, 0.0, 0.0, 0.0)])
    red_patch = mpatches.Patch(color="red", label="Invalid pixels")
    fg = dem["height"].plot(
        ax=ax,
        vmin=dem["height"].min(),
        vmax=dem["height"].max(),
        add_colorbar=False,
    )
    valid.plot(ax=ax, cmap=valid_cmap, add_colorbar=False)
    plt.legend(handles=[red_patch], loc="lower right")
    plt.title(
        f"{'/'.join(tile_ids)} - Slope < {slope_threshold} - Range = {range_threshold} {' - cleaning ' if simplify else ''}\n Percentage of valid pixels = {pvalid * 100:.2f}",
        fontsize=18,
    )
    ax.tick_params(axis="both", which="major", labelsize=12)
    ax.set_xlabel("x", fontsize=15)
    ax.set_ylabel("y", fontsize=15)
    cb = plt.colorbar(fg, orientation="vertical")
    cb.set_label(label="Height (m)", size=17)
    cb.ax.tick_params(labelsize=12)
    plt.legend(handles=[red_patch], loc="lower right", fontsize=16)
    plt.show()

In [ ]:
plot_valid_zones(
    ["31TCJ"],
    land,
    slope_threshold=30,
    range_threshold=600,
)

In [ ]:
def plot_zone_figs(
    tile_ids: list[str],
    land: gpd.GeoDataFrame,
    tiles: gpd.GeoDataFrame,
    loc="lower right",
) -> None:
    """
    Plot valid pixels
    """
    simplify = True
    # Read DEM
    dem = get_dem_from_tiles(tile_ids)
    # Compute various case
    results = []
    for slope_threshold in [15, 30]:
        for range_threshold in [300, 600]:
            for poly_simplify in [600]:
                # Compute valid pixels
                pvalid, nbvalid, polys = zones.define_valid_zones(
                    tile_ids,
                    land,
                    slope_threshold=slope_threshold,
                    range_threshold=range_threshold,
                    simplify=simplify,
                    poly_simplify=poly_simplify,
                )
                # Transform polygons to mask
                valid = xr.DataArray(
                    geometry_mask(
                        polys.geoms,
                        out_shape=(dem.sizes["y"], dem.sizes["x"]),
                        transform=dem.transform,
                    ),
                    coords=dem.coords,
                    dims=("y", "x"),
                )
                valid.data = ~valid.data
                res = valid
                # Save result: slope, range, pvalid and valid
                results.append(
                    [
                        slope_threshold,
                        range_threshold,
                        poly_simplify,
                        pvalid,
                        res.copy(),
                    ]
                )

    # Plot
    nb = len(results)
    fig, axs = plt.subplots(
        ncols=nb + 2, nrows=1, figsize=(6 * (nb + 2), 4), layout="constrained"
    )
    valid_cmap = ListedColormap([(1.0, 0.0, 0.0, 1.0), (0.0, 0.0, 0.0, 0.0)])
    red_patch = mpatches.Patch(color="red", label="Invalid pixels")

    # Location
    gdf = tiles[tiles["id"].isin(tile_ids)]
    fg = gdf.plot(
        ax=axs[0],
        alpha=0.8,
        edgecolor="k",
    )
    bb = gdf.bounds
    axs[0].set(
        xlim=(bb.minx.values[0] - 6.2, bb.maxx.values[0] + 6.2),
        ylim=(bb.miny.values[0] - 3.5, bb.maxy.values[0] + 3.5),
    )
    # ax.set(xlim=(bb.minx.values[0], bb.maxx.values[0]), ylim=(bb.miny.values[0], bb.maxy.values[0]))
    cx.add_basemap(fg, zoom=5, crs=tiles.crs)
    axs[0].tick_params(axis="both", which="major", labelsize=13)
    axs[0].set_ylabel("Latitude", fontsize=15)
    axs[0].set_xlabel("Longitude", fontsize=15)
    axs[0].set_title("Location", fontsize=18)
    # DEM
    ax = axs[1]
    dem["height"].plot(
        ax=ax,
        vmin=dem["height"].min(),
        vmax=dem["height"].max(),
        add_colorbar=False,
    )
    ax.set_title("DEM", fontsize=18)
    ax.tick_params(axis="both", which="major", labelsize=13)
    ax.set_xlabel("x", fontsize=15)
    ax.set_ylabel("y", fontsize=15)
    # Result
    for i, (ax, res) in enumerate(zip(axs[2:], results, strict=False)):
        fg = dem["height"].plot(
            ax=ax,
            vmin=dem["height"].min(),
            vmax=dem["height"].max(),
            add_colorbar=False,
        )
        res[4].plot(ax=ax, cmap=valid_cmap, add_colorbar=False)
        ax.set_title(
            f"Slope < {res[0]} - Range = {res[1]}m{' - simplification ' if res[2] > 0 else ''} \n Percentage of valid pixels = {res[3] * 100:.2f}",
            fontsize=18,
        )
        ax.tick_params(axis="both", which="major", labelsize=13)
        ax.set_xlabel("x", fontsize=15)
        ax.set_ylabel("y", fontsize=15)
        ax.legend(handles=[red_patch], loc=loc, fontsize=16)
        if i == nb - 1:
            cb = plt.colorbar(fg, orientation="vertical")
            cb.set_label(label="Height (m)", size=17)
            cb.ax.tick_params(labelsize=13)

    # fig.suptitle(f"{'/'.join(tile_ids)}", fontsize='xx-large')
    fig.savefig(os.path.join(outdir, f"plot_{'-'.join(tile_ids)}.png"), dpi=600)
    plt.show()

In [ ]:
plot_zone_figs(["32TLP"], land, tiles)

In [ ]:
plot_zone_figs(["32TPQ"], land, tiles, "upper right")